In [1]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 9.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.8 MB/s eta 0:00:00


In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_SHAPE, MSO_CONNECTOR
from pptx.dml.color import RGBColor
from pptx.oxml.ns import nsdecls
from pptx.oxml import parse_xml

# --- Funções Auxiliares ---

def adicionar_seta(slide, shape_origem, shape_destino):
    """Cria uma seta conectando duas formas."""
    conector = slide.shapes.add_connector(
        MSO_CONNECTOR.ELBOW, 0, 0, 0, 0
    )
    # Conecta o início e o fim da seta nas formas
    conector.begin_connect(shape_origem, 2) # idx muda o ponto de conexão (0=cima, 1=dir, 2=baixo, 3=esq)
    conector.end_connect(shape_destino, 0)

    # Hack para adicionar ponta de seta (a lib padrão às vezes cria linhas simples)
    line = conector.line
    line.width = Pt(2)
    line._get_or_add_ln().append(parse_xml(
        f'<a:headEnd type="arrow" w="med" len="med" {nsdecls("a")}/>'
    ))

def criar_slide_tela(prs, titulo, caminho_imagem, descricao_texto):
    """
    Cria um slide com Título, Imagem à esquerda e Texto à direita.
    """
    slide = prs.slides.add_slide(prs.slide_layouts[5]) # Layout 5 = Title Only (vazio)

    # 1. Adicionar Título
    title = slide.shapes.title
    title.text = titulo

    # 2. Adicionar Imagem (Esquerda)
    # Ajuste as posições (Inches) conforme necessário
    top_img = Inches(2.0)
    left_img = Inches(0.5)
    height_img = Inches(4.5) # Define altura, largura é auto-proporcional
    try:
        slide.shapes.add_picture(caminho_imagem, left_img, top_img, height=height_img)
    except FileNotFoundError:
        print(f"AVISO: Imagem não encontrada em {caminho_imagem}. Pulando imagem.")

    # 3. Adicionar Texto de Descrição (Direita)
    left_txt = Inches(6.0)
    top_txt = Inches(2.0)
    width_txt = Inches(4.0)
    height_txt = Inches(4.5)

    txBox = slide.shapes.add_textbox(left_txt, top_txt, width_txt, height_txt)
    tf = txBox.text_frame
    tf.word_wrap = True

    p = tf.add_paragraph()
    p.text = descricao_texto
    p.font.size = Pt(18)

# --- Script Principal ---

# Criar apresentação
prs = Presentation()

# 1. SLIDE DE CAPA
slide_capa = prs.slides.add_slide(prs.slide_layouts[0]) # Layout 0 = Title Slide
slide_capa.shapes.title.text = "Fluxo do Sistema"
slide_capa.placeholders[1].text = "Gerado automaticamente via Python"

# 2. SLIDES DAS TELAS (Exemplo baseados no seu pedido)
# Substitua 'imagem1.png' pelos arquivos reais das suas telas
desc_login = (
    "Tela de Login:\n"
    "- O usuário insere e-mail e senha.\n"
    "- Validação de campos obrigatórios.\n"
    "- Botão 'Esqueci minha senha' para recuperação."
)
criar_slide_tela(prs, "1. Tela de Login", "imagem_login.png", desc_login)

desc_dashboard = (
    "Tela de Dashboard:\n"
    "- Visão geral dos indicadores.\n"
    "- Gráfico de vendas mensal.\n"
    "- Atalhos rápidos para cadastro."
)
criar_slide_tela(prs, "2. Dashboard Principal", "imagem_dashboard.png", desc_dashboard)

# 3. SLIDE DE FLUXOGRAMA (Criação manual das formas)
slide_fluxo = prs.slides.add_slide(prs.slide_layouts[5])
slide_fluxo.shapes.title.text = "Fluxograma de Funcionamento"

# Definir posições iniciais
left_start = Inches(1)
top_start = Inches(2)
w_shape = Inches(2)
h_shape = Inches(1)
gap = Inches(0.5)

# Forma 1: Início (Login)
shape_login = slide_fluxo.shapes.add_shape(
    MSO_SHAPE.ROUNDED_RECTANGLE, left_start, top_start, w_shape, h_shape
)
shape_login.text = "Login Usuário"

# Forma 2: Decisão (Sucesso?)
shape_decisao = slide_fluxo.shapes.add_shape(
    MSO_SHAPE.DIAMOND, left_start + w_shape + gap, top_start, w_shape, h_shape
)
shape_decisao.text = "Dados Válidos?"

# Forma 3: Sucesso (Dashboard)
shape_dash = slide_fluxo.shapes.add_shape(
    MSO_SHAPE.RECTANGLE, left_start + (w_shape + gap)*2, top_start, w_shape, h_shape
)
shape_dash.text = "Acessa Dashboard"

# Forma 4: Erro (Mensagem)
shape_erro = slide_fluxo.shapes.add_shape(
    MSO_SHAPE.RECTANGLE, left_start + w_shape + gap, top_start + h_shape + gap, w_shape, h_shape
)
shape_erro.text = "Exibe Erro"

# Conectar as formas
adicionar_seta(slide_fluxo, shape_login, shape_decisao)
adicionar_seta(slide_fluxo, shape_decisao, shape_dash) # Sim
adicionar_seta(slide_fluxo, shape_decisao, shape_erro) # Não

# Salvar (assume execução remota no Colab)
import os
# Sempre salva em /content quando no Colab remoto
output_path = '/content/apresentacao_sistema.pptx'
# Opcional: se quiser também salvar direto no Google Drive, defina SAVE_TO_DRIVE = True e ajuste DRIVE_DEST
SAVE_TO_DRIVE = False
DRIVE_DEST = '/content/drive/MyDrive/apresentacao_sistema.pptx'
# Garante que o diretório exista
try:
    os.makedirs(os.path.dirname(output_path) or '/content', exist_ok=True)
except Exception as e:
    print(f'Não foi possível garantir diretório para {output_path}: {e}')
# Salva o arquivo em /content
prs.save(output_path)
print(f'Apresentação criada em: {output_path}')
# Tenta iniciar download automático pelo navegador
try:
    from google.colab import files  # type: ignore
    print('Tentando iniciar download automático...')
    files.download(output_path)
except Exception as e:
    print('Download automático falhou ou não suportado no runtime:', e)
    print(f"Execute manualmente no Colab: from google.colab import files; files.download('{output_path}')")
# Se o usuário optou por salvar no Drive, monta e salva lá também
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        prs.save(DRIVE_DEST)
        print(f'Salvo também em Drive: {DRIVE_DEST}')
    except Exception as e:
        print('Falha ao montar/salvar no Drive:', e)
# Instruções finais para mover para seu diretório local após download
print('Após o download no navegador, mova o arquivo para:')
print('/home/vinicius/Downloads/estudo/engenharia-software/gestao-notas-mobile/mobile/docs/')
print('Exemplo no seu computador:')
print("mv ~/Downloads/apresentacao_sistema.pptx /home/vinicius/Downloads/estudo/engenharia-software/gestao-notas-mobile/mobile/docs/")

AVISO: Imagem não encontrada em imagem_login.png. Pulando imagem.
AVISO: Imagem não encontrada em imagem_dashboard.png. Pulando imagem.


FileNotFoundError: [Errno 2] No such file or directory: '/home/vinicius/Downloads/estudo/engenharia-software/gestao-notas-mobile/mobile/docs/apresentacao_sistema.pptx'

In [ ]:
# Preparar imagens para uso no Colab remoto
# Este bloco copia imagens para /content/imgs-app a partir de duas opções:
# 1) Upload manual pelo navegador (files.upload)
# 2) Montagem do Google Drive e leitura de uma pasta no Drive

import os
from typing import List

# Caminhos locais desejados no seu PC (apenas referência; Colab remoto não acessa)
LOCAL_REFERENCE_DIR = "/home/vinicius/Downloads/estudo/engenharia-software/gestao-notas-mobile/mobile/docs/imgs-app"
LOCAL_REFERENCE_IMAGES = [
    "01-notas-fiscais-lista.jpeg",
    "02-nota-fiscal-detalhes.jpeg",
    "03-calendario-mes-lancamentos.jpeg",
    "04-classificar-notas-kanban.jpeg",
    "05-dashboard-header-e-botoes.jpeg",
    "06-dashboard-alertas-e-cards.jpeg",
    "07-notas-fiscais-recentes-card.jpeg",
]

# Diretório de destino no Colab
COLAB_IMG_DIR = "/content/imgs-app"
os.makedirs(COLAB_IMG_DIR, exist_ok=True)

# MODO DE OBTENÇÃO DAS IMAGENS
#   "upload"  -> usa files.upload() para enviar a partir do seu computador
#   "drive"   -> monta o Google Drive e copia de uma pasta configurada lá
IMAGE_MODE = "upload"  # altere para "drive" se preferir

# Se usar DRIVE, ajuste de onde copiar dentro do Drive
DRIVE_SRC_DIR = "/content/drive/MyDrive/gestao-notas-mobile/imgs-app"  # ajuste conforme necessário

resolved_paths: List[str] = []

if IMAGE_MODE == "upload":
    try:
        from google.colab import files  # type: ignore
        print("Selecione suas imagens para upload (você pode marcar múltiplos arquivos)...")
        uploaded = files.upload()  # abre seletor do navegador
        for fname in uploaded.keys():
            src = f"/content/{fname}"
            dst = os.path.join(COLAB_IMG_DIR, fname)
            try:
                os.replace(src, dst)  # move para a pasta destino
                resolved_paths.append(dst)
            except Exception as e:
                print(f"Falha ao mover {src} -> {dst}: {e}")
        print(f"Imagens copiadas para: {COLAB_IMG_DIR}")
    except Exception as e:
        print("Upload não disponível neste runtime:", e)
        print("Alternativa: mude IMAGE_MODE para 'drive' e use Google Drive.")

elif IMAGE_MODE == "drive":
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        # Copia todos os arquivos esperados da pasta do Drive
        for fname in LOCAL_REFERENCE_IMAGES:
            src = os.path.join(DRIVE_SRC_DIR, fname)
            dst = os.path.join(COLAB_IMG_DIR, fname)
            try:
                # Usa leitura binária e escrita para evitar dependência de !cp
                with open(src, 'rb') as rf, open(dst, 'wb') as wf:
                    wf.write(rf.read())
                resolved_paths.append(dst)
            except FileNotFoundError:
                print(f"Arquivo não encontrado no Drive: {src}")
            except Exception as e:
                print(f"Falha ao copiar {src} -> {dst}: {e}")
        print(f"Imagens copiadas para: {COLAB_IMG_DIR}")
    except Exception as e:
        print('Falha ao montar/copiar do Drive:', e)
        print('Verifique o caminho DRIVE_SRC_DIR ou permissões de acesso.')
else:
    print("IMAGE_MODE inválido. Use 'upload' ou 'drive'.")

# Helper: mapeia nomes esperados para paths resolvidos dentro do Colab
# Exemplo de uso: caminho_imagem = get_image_path("01-notas-fiscais-lista.jpeg")

def get_image_path(filename: str) -> str:
    candidate = os.path.join(COLAB_IMG_DIR, filename)
    if os.path.exists(candidate):
        return candidate
    # fallback: se não existir, retorna o próprio nome (será tratado pela função criar_slide_tela)
    return filename

print("Imagens disponíveis:")
for p in sorted(os.listdir(COLAB_IMG_DIR)):
    print(os.path.join(COLAB_IMG_DIR, p))
